<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>04. 💬 TCP Socket Chat System</b>
</div>


### 📌 Project Overview
In this project, we build an asynchronous multi-user chat server and client application using low-level network sockets.
Using Python's standard `socket` module to configure TCP/IP communication and `threading` to support concurrent user sessions, students learn the fundamentals of network protocols and concurrent stream reading.

#### 🔑 Key Concepts Covered:
- Creating, binding, and listening on TCP IPv4 sockets via `socket.socket()`
- Dispatching background connection workers dynamically using `threading.Thread`
- Broad-spectrum message broadcasting across an active sockets hash dictionary
- Handling unexpected client disconnections, stream timeouts, and socket close actions


In [ ]:
import socket
import threading
import time

# Chat System Local Config
HOST = '127.0.0.1'
PORT = 0  # 0 tells the OS to assign any free available port dynamically
PORT_ASSIGNED = None
active_clients = {}

def broadcast_payload(msg_text, exclude_conn=None):
    """Transmits message to all connected clients."""
    for conn in list(active_clients.keys()):
        if conn != exclude_conn:
            try:
                conn.sendall(msg_text.encode('utf-8'))
            except socket.error:
                disconnect_client(conn)

def service_client(conn, addr):
    """Listens to messages from a single client connection."""
    print(f'[SERVER] Connection accepted from {addr}')
    try:
        # Register client username
        name = conn.recv(1024).decode('utf-8').strip()
        if not name:
            name = f'Guest_{addr[1]}'
        active_clients[conn] = name
        broadcast_payload(f'[JOIN] {name} has joined the room!')
        
        while True:
            data = conn.recv(1024)
            if not data:
                break
            message = data.decode('utf-8').strip()
            print(f'[SERVER] {name}: {message}')
            broadcast_payload(f'{name}: {message}', exclude_conn=conn)
            
    except socket.error as e:
        print(f'[SERVER ERROR] Connection error with {addr}: {e}')
    finally:
        disconnect_client(conn)

def disconnect_client(conn):
    """Closes a connection and deletes it from list."""
    if conn in active_clients:
        name = active_clients[conn]
        print(f'[SERVER] {name} left.')
        del active_clients[conn]
        try:
            conn.close()
        except socket.error:
            pass
        broadcast_payload(f'[LEAVE] {name} has left the room.')

def initialize_chat_server():
    """Starts listening socket for up to 2 client interactions."""
    global PORT_ASSIGNED
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        server.bind((HOST, PORT))
        PORT_ASSIGNED = server.getsockname()[1]
        server.listen()
        print(f'[SERVER] Chat room server hosted at {HOST}:{PORT_ASSIGNED}')
        
        # Simple demo counter to close server after 2 clients disconnect
        for _ in range(2):
            conn, addr = server.accept()
            client_thread = threading.Thread(target=service_client, args=(conn, addr))
            client_thread.daemon = True
            client_thread.start()
    except Exception as exc:
        print(f'[SERVER EXCEPTION] {exc}')
    finally:
        server.close()
        print('[SERVER] Closed server socket.')


In [ ]:
def run_mock_client(client_name, messages):
    """Simulates a connecting chat client."""
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        sock.connect((HOST, PORT_ASSIGNED))
        # Send username
        sock.sendall(client_name.encode('utf-8'))
        time.sleep(0.1)
        
        # Spawn receiver thread
        def reader():
            while True:
                try:
                    incoming = sock.recv(1024).decode('utf-8')
                    if not incoming:
                        break
                    print(f'[{client_name} Screen] {incoming}')
                except socket.error:
                    break
                    
        t = threading.Thread(target=reader)
        t.daemon = True
        t.start()
        
        for msg in messages:
            time.sleep(0.4)
            sock.sendall(msg.encode('utf-8'))
            
        time.sleep(0.8)
    except Exception as e:
        print(f'[{client_name} Error]: {e}')
    finally:
        sock.close()

# 1. Start Server thread
PORT_ASSIGNED = None  # Reset before starting
server_thread = threading.Thread(target=initialize_chat_server)
server_thread.daemon = True
server_thread.start()

# Wait for server to bind and assign port
for _ in range(50):
    if PORT_ASSIGNED is not None:
        break
    time.sleep(0.1)

if PORT_ASSIGNED is None:
    print('[SERVER ERROR] Failed to bind server.')
else:
    # 2. Connect client Alice
    alice_thread = threading.Thread(
        target=run_mock_client, 
        args=('Alice', ['Hi everyone!', 'Happy Coding!', 'Signing off!'])
    )
    alice_thread.start()
    
    # 3. Connect client Bob
    time.sleep(0.2)
    bob_thread = threading.Thread(
        target=run_mock_client, 
        args=('Bob', ['Hello Alice!', 'I love Python networking!', 'Goodbye!'])
    )
    bob_thread.start()
    
    alice_thread.join()
    bob_thread.join()
    print('\n[SIMULATION COMPLETE] Socket chat demo finalized.')
